# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Install mlcroissant if not already available
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# Present dataset metadata
meta = dataset.metadata
print(f"{meta.name}: {meta.description}\n")
print(f"Published: {meta.datePublished}")
print(f"Dataset Identifier: {meta.identifier}")
print(f"Authors: {meta.author if hasattr(meta, 'author') else 'N/A'}")

## 2. Data Overview
Review available record sets, their fields, and `@id`s.

The Croissant dataset organizes tabular data into one or more *record sets* (think: one per table/file). Each record set contains a set of fields (columns), each with a unique `@id` for precise reference.

In [ ]:
# Examine the record sets available in this dataset
record_sets = dataset.record_sets
print(f"Found {len(record_sets)} record set(s).\n")

for rs in record_sets:
    print(f"Record Set: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Number of Fields: {len(rs.fields)}")
    print("  Field IDs and Names:")
    for field in rs.fields:
        print(f"    - {field.id}: {field.name} (type: {field.data_type})")
    print()

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Use the record set and field `@id`s identified above.

Note: If there are multiple record sets (tables), they will each be loaded into a DataFrame keyed by their `@id`.

In [ ]:
# List all record set @id values for reference
record_set_ids = [rs.id for rs in dataset.record_sets]

dataframes = {}
for rs in dataset.record_sets:
    print(f"Loading records for record set {rs.id} ({rs.name}) ...")
    # Load records (as list of dicts)
    records = list(dataset.records(record_set=rs.id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs.id] = df
        print(f"  -> Loaded {len(df)} records. Columns: {list(df.columns)}\n")
    else:
        print("  -> No records found.\n")

# Show the columns of the first record set loaded (choose the first present)
# (If more than one, demonstration proceeds with the first)
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    print(f"Columns in record set {first_rs_id}:\n{dataframes[first_rs_id].columns.tolist()}")
    display(dataframes[first_rs_id].head())
else:
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data for further analysis.

### Example: Filter, normalize, group
- Select a numeric field/column by its `@id` that represents a measurement (e.g., 'cr:coveragePercent' or 'cr:molecularWeight' if available).
- Filter rows where the value is above a threshold.
- Normalize that field.
- Group by a categorical field (e.g., 'cr:sampleName') if available.

In [ ]:
# Find a numeric field and a group field in the first loaded record set
# (This demonstration assumes typical fields like 'cr:coveragePercent', 'cr:peptideCount', 'cr:molecularWeight' exist)
rs = dataset.record_sets[0] if dataset.record_sets else None
df = dataframes[rs.id] if rs and rs.id in dataframes else None

if rs and df is not None:
    # Find candidate numeric fields
    numeric_fields = [f.id for f in rs.fields if f.data_type in ('Float', 'Integer', 'Number') and f.id in df.columns]
    group_fields = [f.id for f in rs.fields if f.data_type in ('Text', 'String') and f.id in df.columns]

    print(f"Numeric fields detected: {numeric_fields}")
    print(f"Text (group) fields detected: {group_fields}")

    # Choose the first available numeric field for demonstration
    numeric_field = numeric_fields[0] if numeric_fields else None
    group_field = group_fields[0] if group_fields else None

    if numeric_field:
        # Handle missing data and type conversion for robustness
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        # Threshold: use mean as default, or 10 if mean is invalid
        threshold = df[numeric_field].mean() if not pd.isnull(df[numeric_field].mean()) else 10.0
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f} (total {len(filtered_df)} rows):")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"\nNormalized '{numeric_field}' for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group if a group field exists
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame(name=f"mean_{numeric_field}")
            print(f"\nGrouped (mean) '{numeric_field}' by '{group_field}':")
            display(grouped_df.head())
        else:
            print('No suitable group field found for grouping.')
    else:
        print("No numeric fields found for EDA.")
else:
    print("No data loaded or available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

For example, plot the distribution of a numeric field, or compare group means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field:
    # Histogram of full numeric field
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    # Boxplot by group if group_field exists and is not too unique
    if group_field and df[group_field].nunique() < 20:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Visualization unavailable; no numeric field detected.")

## 6. Conclusion
In this notebook, we demonstrated how to load, explore, and process the FAIR\textasciicircum2 dataset using its Croissant schema and the `mlcroissant` library,
referencing all components by their `@id`. We conducted basic Exploratory Data Analysis, such as filtering, normalization, grouping, and data visualization.

Key steps included:
- Listing record sets and column (field) `@id`s
- Loading the data from each set as a DataFrame
- Filtering and normalizing a numeric variable
- Grouping and plotting summaries

This workflow can serve as a starting point for further, domain-specific analysis or integration with other biological or proteomic resources.